# Ch.6 — Cold Start & Production

> Handle new users with zero history using bandit exploration. Build a production recommendation pipeline with A/B testing and monitoring.

**Dataset:** MovieLens 100k  
**Task:** Implement cold start strategies (bandits) + production architecture  
**Outcome:** All 5 FlixAI constraints satisfied ✅

## The Core Idea

**Cold Start Problem**: New users have zero interaction history → no collaborative embedding → what to recommend?

**Solutions**:
1. **Content-based fallback**: Use genre preferences from onboarding
2. **Bandit exploration**: Balance safe (exploit) vs. uncertain (explore) recommendations  
3. **Gradual transition**: Cold start → warm-up → full hybrid model

**Production concerns**: A/B testing, serving latency, retraining, monitoring.

**Epsilon-Greedy**: $a_t = \arg\max \hat{r}(a)$ with probability $1 - \epsilon$, random with probability $\epsilon$

**UCB**: $\text{UCB}(a) = \hat{r}(a) + c\sqrt{\frac{\ln t}{T_a}}$

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
np.random.seed(SEED)

print("Libraries loaded.")

In [ ]:
# ── Load MovieLens 100k ───────────────────────────────────────────────────
url = "https://files.grouplens.org/datasets/movielens/ml-100k/"

ratings = pd.read_csv(
    url + "u.data", sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"]
)

movies = pd.read_csv(
    url + "u.item", sep="|", encoding="latin-1", header=None,
    names=["item_id", "title", "release_date", "video_release", "url",
           "unknown", "Action", "Adventure", "Animation", "Children",
           "Comedy", "Crime", "Documentary", "Drama", "Fantasy",
           "Film-Noir", "Horror", "Musical", "Mystery", "Romance",
           "Sci-Fi", "Thriller", "War", "Western"],
    usecols=range(24)
)

genre_cols = ["Action", "Adventure", "Animation", "Children",
              "Comedy", "Crime", "Documentary", "Drama", "Fantasy",
              "Film-Noir", "Horror", "Musical", "Mystery", "Romance",
              "Sci-Fi", "Thriller", "War", "Western"]

n_users = ratings["user_id"].max()
n_items = ratings["item_id"].max()

print(f"Users: {n_users}  Items: {n_items}  Ratings: {len(ratings):,}")

In [ ]:
# ── Simulate Cold Start Scenario ──────────────────────────────────────────
# Pick 100 users as "new" users: hold out ALL their ratings
np.random.seed(SEED)
cold_users = np.random.choice(range(1, n_users + 1), size=100, replace=False)

cold_ratings = ratings[ratings["user_id"].isin(cold_users)].copy()
warm_ratings = ratings[~ratings["user_id"].isin(cold_users)].copy()

# Simulate onboarding: cold users' first 3 ratings are "onboarding"
cold_sorted = cold_ratings.sort_values("timestamp")
onboarding = cold_sorted.groupby("user_id").head(3)
cold_test = cold_sorted.groupby("user_id").tail(1)

print(f"Cold start users: {len(cold_users)}")
print(f"Onboarding ratings: {len(onboarding)} (3 per user)")
print(f"Cold test ratings: {len(cold_test)} (1 per user)")
print(f"Warm training ratings: {len(warm_ratings):,}")

In [ ]:
# ── Content-Based Fallback ────────────────────────────────────────────────
def content_based_recommend(user_genre_prefs, item_genres_df, genre_cols, n_recs=10):
    """Recommend items based on genre preference similarity."""
    item_features = item_genres_df[genre_cols].values.astype(float)
    user_vec = np.array(user_genre_prefs, dtype=float)
    
    if user_vec.sum() == 0:
        # No genre info: fall back to popular
        return list(range(1, n_recs + 1))
    
    # Cosine similarity between user preference and each item
    scores = item_features @ user_vec
    norms = np.linalg.norm(item_features, axis=1) * np.linalg.norm(user_vec)
    norms[norms == 0] = 1
    scores = scores / norms
    
    top_items = np.argsort(scores)[-n_recs:][::-1]
    return (item_genres_df.iloc[top_items]["item_id"].values if "item_id" in item_genres_df.columns 
            else top_items + 1).tolist()

# Test: build genre preferences from onboarding data
def get_genre_prefs_from_ratings(user_ratings, movies_df, genre_cols):
    """Derive genre preferences from a user's rated movies (weighted by rating)."""
    merged = user_ratings.merge(movies_df[["item_id"] + genre_cols], on="item_id")
    genre_scores = np.zeros(len(genre_cols))
    for _, row in merged.iterrows():
        genre_scores += row[genre_cols].values.astype(float) * row["rating"]
    return genre_scores / max(genre_scores.sum(), 1)

# Example for one cold user
sample_user = cold_users[0]
sample_onboard = onboarding[onboarding["user_id"] == sample_user]
prefs = get_genre_prefs_from_ratings(sample_onboard, movies, genre_cols)
print(f"User {sample_user} genre preferences from {len(sample_onboard)} onboarding ratings:")
for g, p in zip(genre_cols, prefs):
    if p > 0.01:
        print(f"  {g}: {p:.2f}")

## Bandit Algorithms for Exploration

### Epsilon-Greedy
With probability $\epsilon$, explore (random); otherwise exploit (model's best).

### Upper Confidence Bound (UCB)
$$\text{UCB}(a) = \hat{r}(a) + c\sqrt{\frac{\ln t}{T_a}}$$

Prefers items with high predicted score OR high uncertainty (rarely recommended).

In [ ]:
# ── Epsilon-Greedy Bandit ─────────────────────────────────────────────────
class EpsilonGreedyBandit:
    def __init__(self, n_items, epsilon=0.3, decay=0.95, min_epsilon=0.05):
        self.n_items = n_items
        self.epsilon = epsilon
        self.decay = decay
        self.min_epsilon = min_epsilon
        self.item_rewards = np.zeros(n_items + 1)
        self.item_counts = np.zeros(n_items + 1)
    
    def select(self, model_scores, already_rated, n_recs=10):
        """Select items balancing exploitation and exploration."""
        model_scores = model_scores.copy()
        for r in already_rated:
            if r < len(model_scores):
                model_scores[r] = -np.inf
        model_scores[0] = -np.inf
        
        if np.random.random() < self.epsilon:
            # Explore: top-7 from model + 3 random
            n_exploit = max(1, n_recs - 3)
            n_explore = n_recs - n_exploit
            top = np.argsort(model_scores)[-n_exploit:][::-1]
            
            valid = np.where(model_scores > -np.inf)[0]
            explore = np.random.choice(valid, size=min(n_explore, len(valid)), replace=False)
            return np.concatenate([top, explore]).astype(int).tolist()[:n_recs]
        else:
            return np.argsort(model_scores)[-n_recs:][::-1].tolist()
    
    def update(self, item_id, reward):
        self.item_counts[item_id] += 1
        n = self.item_counts[item_id]
        self.item_rewards[item_id] += (reward - self.item_rewards[item_id]) / n
    
    def decay_epsilon(self):
        self.epsilon = max(self.min_epsilon, self.epsilon * self.decay)

print("EpsilonGreedyBandit defined.")

In [ ]:
# ── UCB Bandit ────────────────────────────────────────────────────────────
class UCBBandit:
    def __init__(self, n_items, c=1.5):
        self.n_items = n_items
        self.c = c
        self.item_rewards = np.zeros(n_items + 1)
        self.item_counts = np.zeros(n_items + 1)
        self.total_pulls = 0
    
    def select(self, model_scores, already_rated, n_recs=10):
        """Select items using UCB: predicted score + exploration bonus."""
        self.total_pulls += 1
        
        ucb_scores = model_scores.copy()
        for i in range(1, self.n_items + 1):
            if self.item_counts[i] > 0:
                exploration = self.c * np.sqrt(np.log(self.total_pulls) / self.item_counts[i])
            else:
                exploration = self.c * 10  # high bonus for unexplored
            ucb_scores[i] = model_scores[i] + exploration
        
        for r in already_rated:
            if r < len(ucb_scores):
                ucb_scores[r] = -np.inf
        ucb_scores[0] = -np.inf
        
        return np.argsort(ucb_scores)[-n_recs:][::-1].tolist()
    
    def update(self, item_id, reward):
        self.item_counts[item_id] += 1
        n = self.item_counts[item_id]
        self.item_rewards[item_id] += (reward - self.item_rewards[item_id]) / n

print("UCBBandit defined.")

In [ ]:
# ── Simulate Cold Start with Different Strategies ─────────────────────────
# Strategy 1: Pure popularity (no exploration)
# Strategy 2: Content-based only
# Strategy 3: Epsilon-greedy (ε=0.3)
# Strategy 4: UCB (c=1.5)

# Build popularity scores from warm data
item_stats = warm_ratings.groupby("item_id")["rating"].agg(["mean", "count"])
global_mean = warm_ratings["rating"].mean()
C = item_stats["count"].median()
item_stats["pop_score"] = (item_stats["count"] * item_stats["mean"] + C * global_mean) / (item_stats["count"] + C)

pop_scores = np.zeros(n_items + 1)
for iid, row in item_stats.iterrows():
    pop_scores[iid] = row["pop_score"]

def evaluate_cold_strategy(strategy_name, rec_func, cold_test_df, k=10):
    """Evaluate a cold-start strategy on held-out cold user test items."""
    hits = 0
    for _, row in cold_test_df.iterrows():
        uid = row["user_id"]
        test_item = row["item_id"]
        recs = rec_func(uid)[:k]
        if test_item in recs:
            hits += 1
    return hits / len(cold_test_df)

# Strategy 1: Pure popularity
def pop_strategy(uid):
    rated = set(onboarding[onboarding["user_id"] == uid]["item_id"])
    scores = pop_scores.copy()
    for r in rated:
        scores[r] = -np.inf
    return np.argsort(scores)[-10:][::-1].tolist()

# Strategy 2: Content-based
movies_indexed = movies.set_index("item_id")
def content_strategy(uid):
    user_onboard = onboarding[onboarding["user_id"] == uid]
    prefs = get_genre_prefs_from_ratings(user_onboard, movies, genre_cols)
    item_features = movies[genre_cols].values.astype(float)
    scores = item_features @ prefs
    rated = set(user_onboard["item_id"])
    for r in rated:
        if r - 1 < len(scores):
            scores[r - 1] = -np.inf
    return (np.argsort(scores)[-10:][::-1] + 1).tolist()

# Strategy 3: ε-greedy
bandit_eg = EpsilonGreedyBandit(n_items, epsilon=0.3)
def eg_strategy(uid):
    rated = set(onboarding[onboarding["user_id"] == uid]["item_id"])
    return bandit_eg.select(pop_scores, rated, n_recs=10)

# Strategy 4: UCB
bandit_ucb = UCBBandit(n_items, c=1.5)
def ucb_strategy(uid):
    rated = set(onboarding[onboarding["user_id"] == uid]["item_id"])
    return bandit_ucb.select(pop_scores, rated, n_recs=10)

hr_pop = evaluate_cold_strategy("Popularity", pop_strategy, cold_test)
hr_content = evaluate_cold_strategy("Content", content_strategy, cold_test)
hr_eg = evaluate_cold_strategy("ε-Greedy", eg_strategy, cold_test)
hr_ucb = evaluate_cold_strategy("UCB", ucb_strategy, cold_test)

print(f"Cold Start Strategies (HR@10):")
print(f"  Popularity:    {hr_pop:.3f} ({hr_pop*100:.1f}%)")
print(f"  Content-Based: {hr_content:.3f} ({hr_content*100:.1f}%)")
print(f"  ε-Greedy:      {hr_eg:.3f} ({hr_eg*100:.1f}%)")
print(f"  UCB:           {hr_ucb:.3f} ({hr_ucb*100:.1f}%)")

In [ ]:
# ── Visualise Cold Start Strategies ───────────────────────────────────────
strategies = ["Popularity", "Content-Based", "ε-Greedy", "UCB"]
hr_values = [hr_pop, hr_content, hr_eg, hr_ucb]
colors = ["#95a5a6", "#3498db", "#27ae60", "#e74c3c"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(strategies, [h * 100 for h in hr_values], color=colors, edgecolor="white")
ax.set(ylabel="Hit Rate@10 (%)", title="Cold Start Strategies — New Users (3 onboarding ratings)")
for bar, val in zip(bars, hr_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val*100:.1f}%", ha="center", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("img/cold_start_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Epsilon Decay Simulation ──────────────────────────────────────────────
interactions = range(1, 101)
epsilon_history = []
eps = 0.3
decay = 0.95
min_eps = 0.05

for t in interactions:
    epsilon_history.append(eps)
    eps = max(min_eps, eps * decay)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(interactions, epsilon_history, color="#27ae60", linewidth=2)
ax.axhline(0.05, color="gray", linestyle="--", alpha=0.5, label="Min ε = 0.05")
ax.fill_between(interactions, epsilon_history, alpha=0.1, color="#27ae60")
ax.set(xlabel="Number of Interactions", ylabel="Exploration Rate (ε)",
       title="Epsilon Decay: Exploration → Exploitation")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("img/epsilon_decay.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"ε after 10 interactions: {epsilon_history[9]:.3f}")
print(f"ε after 50 interactions: {epsilon_history[49]:.3f}")
print(f"ε after 100 interactions: {epsilon_history[99]:.3f}")

In [ ]:
# ── A/B Test Power Analysis ───────────────────────────────────────────────
def ab_test_sample_size(p_control, min_lift, alpha=0.05, power=0.8):
    """Compute minimum sample size per group for a two-proportion z-test."""
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power)
    p_treat = p_control + min_lift
    p_pool = (p_control + p_treat) / 2
    n = (z_alpha + z_beta) ** 2 * 2 * p_pool * (1 - p_pool) / min_lift ** 2
    return int(np.ceil(n))

def ab_test_significance(hr_a, n_a, hr_b, n_b, alpha=0.05):
    """Two-proportion z-test."""
    p_pool = (hr_a * n_a + hr_b * n_b) / (n_a + n_b)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n_a + 1/n_b))
    if se == 0:
        return {"z_stat": 0, "p_value": 1.0, "significant": False}
    z = (hr_b - hr_a) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    return {"z_stat": z, "p_value": p_value, "significant": p_value < alpha}

# How many users do we need per group to detect a 2% lift?
n_per_group = ab_test_sample_size(p_control=0.85, min_lift=0.02)
print(f"A/B Test Power Analysis:")
print(f"  Detecting 2% lift from 85% baseline")
print(f"  α=0.05, power=0.80")
print(f"  Required: {n_per_group:,} users per group")
print(f"  Total traffic needed: {n_per_group * 2:,} users")

In [ ]:
# ── Simulate A/B Test ─────────────────────────────────────────────────────
np.random.seed(SEED)

# Simulate: Model A (HR=85%), Model B (HR=87%)
n_sim = 5000
hr_a_true = 0.85
hr_b_true = 0.87

results_a = np.random.binomial(1, hr_a_true, n_sim)
results_b = np.random.binomial(1, hr_b_true, n_sim)

# Running significance test
p_values = []
sample_sizes = range(100, n_sim + 1, 100)

for n in sample_sizes:
    result = ab_test_significance(
        results_a[:n].mean(), n,
        results_b[:n].mean(), n
    )
    p_values.append(result["p_value"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sample_sizes, p_values, color="#2980b9", linewidth=1.5)
ax.axhline(0.05, color="red", linestyle="--", alpha=0.7, label="p = 0.05 threshold")
ax.set(xlabel="Sample Size per Group", ylabel="p-value",
       title="A/B Test: When Does the 2% Lift Become Significant?")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale("log")
plt.tight_layout()
plt.savefig("img/ab_test_pvalue.png", dpi=150, bbox_inches="tight")
plt.show()

# Find when significance is first achieved
for n, p in zip(sample_sizes, p_values):
    if p < 0.05:
        print(f"First significant at n={n} per group (p={p:.4f})")
        break

## Final Progress Check — All Constraints

| # | Constraint | Target | Final Status |
|---|-----------|--------|-------------|
| 1 | ACCURACY | >85% HR@10 | ✅ **87%** (Hybrid DCN, Ch.5) |
| 2 | COLD START | New users/items | ✅ **Bandit + content fallback** |
| 3 | SCALABILITY | 1M+ ratings | ✅ **Two-stage pipeline** |
| 4 | DIVERSITY | Not just popular | ✅ **MMR re-ranking** |
| 5 | EXPLAINABILITY | "Because you liked X" | ✅ **Content features** |

**🏆 GRAND CHALLENGE COMPLETE**: FlixAI is production-ready!

**Journey**: Popularity (42%) → CF (68%) → MF (78%) → NCF (83%) → Hybrid (87%) ✅

## Exercises

**Exercise 1 — LinUCB (Contextual Bandit)**  
Implement LinUCB: use user features (age, gender, genre preferences) as context to personalise exploration. Compare HR@10 against basic UCB for cold-start users.

**Exercise 2 — Warm-Up Simulation**  
Simulate a user's journey from cold start (0 ratings) to established (50 ratings). At each step, record which strategy is used and the cumulative hit rate. Plot the transition.

**Exercise 3 — Monitoring Dashboard**  
Build a monitoring dashboard that tracks: daily HR@10, cold-start user fraction, exploration rate, and recommendation diversity. Use matplotlib to create a 2×2 subplot dashboard.

In [ ]:
# ── Exercise 1 scaffold — LinUCB ─────────────────────────────────────────
# TODO: Implement contextual bandit using user features
# LinUCB: UCB(a|x) = θ_a^T x + c * sqrt(x^T A_a^{-1} x)

# class LinUCBBandit:
#     def __init__(self, n_items, n_features, c=1.5):
#         self.A = [np.eye(n_features) for _ in range(n_items + 1)]
#         self.b = [np.zeros(n_features) for _ in range(n_items + 1)]
#         ...
#     def select(self, user_context, ...):
#         ...

pass

In [ ]:
# ── Exercise 2 scaffold — Warm-Up Simulation ─────────────────────────────
# TODO: Simulate a user going from 0 to 50 interactions
# Track: which strategy used (cold/warm/full), cumulative HR

# for n_interactions in range(0, 51):
#     if n_interactions < 10:
#         strategy = "cold_start"
#     elif n_interactions < 50:
#         strategy = "warm_blend"
#     else:
#         strategy = "full_hybrid"
#     # ... make recommendation, check if hit, record

pass

In [ ]:
# ── Exercise 3 scaffold — Monitoring Dashboard ───────────────────────────
# TODO: Create a 2x2 subplot dashboard showing:
# Top-left: Daily HR@10 over 30 days
# Top-right: Cold-start user fraction
# Bottom-left: Exploration rate (epsilon) over time
# Bottom-right: Recommendation diversity (ILD)

# fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# ... simulate 30 days of metrics
# plt.savefig("img/monitoring_dashboard.png", dpi=150, bbox_inches="tight")

pass